In [1]:
# Extract English words from ConceptNet CSV with specific relationships
import csv
import json
from tqdm import tqdm
import re
import spacy
import numpy as np
import pandas as pd
import pyinflect
nlp = spacy.load("en_core_web_sm")

In [2]:
PARSE_CSV = False
UNWEIGHT_JSON = False
MATRIX = False
CLEAN_NP = True

In [3]:
# Relationships we care about
# grab from the full CSV file downloaded from https://github.com/commonsense/conceptnet5/wiki/Downloads

if PARSE_CSV:
    target_rels = {"/r/UsedFor", "/r/CapableOf", "/r/ReceivesAction"}

    graph = {}

    CSV_FILE = "/Users/mcharit2/Downloads/full_conceptnet_ass.csv"

    # First, count total lines for tqdm progress bar
    with open(CSV_FILE, encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)

    with open(CSV_FILE, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter='\t')
        for row in tqdm(reader, total=total_lines, desc="Processing ConceptNet"):
            if len(row) < 4:
                continue

            rel, uri1, uri2 = row[1], row[2], row[3]  # col0=ID, col1=rel, col2=start, col3=end

            # Only keep target relationships
            if rel not in target_rels:
                continue

            # Only keep if both ends are English
            if uri1.startswith("/c/en/") and uri2.startswith("/c/en/"):
                
                # get the subject
                subject = uri1.split("/")[3].replace("_", " ")
                object = uri2.split("/")[3].replace("_", " ")
                if subject not in graph:
                    graph[subject] = []
                graph[subject].append(object)

    # Save to file
    with open("../bank_files/conceptnet_graph_raw.json", "w", encoding="utf-8") as f:
        json.dump(graph, f, ensure_ascii=False, indent=2)


In [4]:
def get_verb_noun(phrase):
    ''' Extract the main verb and noun from a phrase using spaCy '''
    doc_obj = nlp(phrase)
    verb = None
    noun = None
    for token in doc_obj:
        if token.dep_ == "compound" and token.head.pos_ == "NOUN":      # handle compound nouns
            noun = f"{token.text} {token.head.text}"

        elif token.pos_ == "NOUN" and not noun:             # take the first noun we see
            noun = token.lemma_
            
        elif token.pos_ == "VERB":                      # take the first verb we see
            # verb = token.lemma_
            verb = token._.inflect("VBD") if token._.inflect("VBD") else token.lemma_       # convert to past tense if possible
            

        if verb and noun:
            break
    return verb, noun

In [5]:
# parse the raw graph into a more structured format based on conceptnet_exp into noweight format
# exported as a JSON

if UNWEIGHT_JSON:
    graph = json.load(open("../bank_files/conceptnet_graph_raw.json", "r", encoding="utf-8"))
    word_graph = {}
    for subj, obj_list in tqdm(graph.items(), desc="Parsing graph into word graph"):
        if " " in subj:
            continue  # skip multi-word subjects
        if subj not in word_graph:
            word_graph[subj] = {}
        for obj in obj_list:
            # use spacy to determine pos in obj
            verb, noun = get_verb_noun(obj)

            if verb and noun:
                if verb not in word_graph[subj]:
                    word_graph[subj][verb] = []
                if noun not in word_graph[subj][verb]:
                    word_graph[subj][verb].append(noun)

    # save to file
    with open("../bank_files/full_word_graph_noweight.json", "w", encoding="utf-8") as f:
        json.dump(word_graph, f, ensure_ascii=False, indent=2)


In [6]:
# create verb, subject, and object lists to export
if UNWEIGHT_JSON:
    word_graph = json.load(open("../bank_files/full_word_graph_noweight.json"))
    VERBS = []
    SUBJECTS = []
    OBJECTS = []
    for subj, verb_dict in word_graph.items():
        SUBJECTS.append(subj)
        for verb, obj_list in verb_dict.items():
            VERBS.append(verb)
            for obj in obj_list:
                OBJECTS.append(obj)

    VERBS = list(set(VERBS))
    SUBJECTS = list(set(SUBJECTS))
    OBJECTS = list(set(OBJECTS))

    VERBS.sort()
    SUBJECTS.sort()
    OBJECTS.sort()

    np.save("../bank_files/VERBS.npy", np.array(VERBS))
    np.save("../bank_files/SUBJECTS.npy", np.array(SUBJECTS))
    np.save("../bank_files/OBJECTS.npy", np.array(OBJECTS))


In [7]:
# create a 2D matrix of directed graph relationships between nouns with the entries being a list of verbs
# row: subject noun
# col: object noun

# index: verb 

if MATRIX:
    graph = json.load(open("../bank_files/conceptnet_graph_raw.json", "r", encoding="utf-8"))
    word_graph = {}
    

    subject_list = []
    object_list = []
    verb_index = {}

    # lemmatize the subject and objects and grab the verb and noun from the object phrase
    for s, phrases in tqdm(graph.items(), desc="Parsing graph into word graph"):
        # use spacy to determine pos in subject and make sure it's a noun
        nlp_subj = nlp(s)
        tok = nlp_subj[0]
        if tok.pos_ == "NOUN":
            subj = tok.lemma_
        else:
            continue  # skip non-noun subjects

        if len(subj) < 3 or not re.match("^[a-zA-Z]+$", subj):
            continue  # skip multi-word subjects

        if subj not in word_graph:          # initialize subject in word graph if not already there
            word_graph[subj] = {}

        if subj not in subject_list:        # add to subject list if not already there
            subject_list.append(subj)

        # break down phrases into verb and noun
        for obj in phrases:
            # use spacy to determine pos in obj
            verb, noun = get_verb_noun(obj)

            if noun and (len(noun) < 3 or not re.match("^[a-zA-Z]+$", noun)):
                continue  # skip invalid nouns

            if verb and noun:
                if verb not in verb_index:
                    verb_index[verb] = len(verb_index)

                if noun not in word_graph[subj]:
                    word_graph[subj][noun] = []
                if verb not in word_graph[subj][noun]:
                    word_graph[subj][noun].append(verb_index[verb])

                
                if noun not in object_list:
                    object_list.append(noun)



In [8]:
if MATRIX:
    print("Creating matrix with {} subjects, {} objects, and {} verbs".format(len(subject_list), len(object_list), len(verb_index)))

    subject_list = sorted(subject_list)
    object_list = sorted(object_list)

    # create the matrix (as a pandas dataframe)
    matrix = pd.DataFrame(index=subject_list, columns=object_list)

    for subj in tqdm(subject_list, desc="Filling matrix"):
        for obj in object_list:
            if obj in word_graph[subj]:
                matrix.loc[subj, obj] = word_graph[subj][obj]
            else:
                matrix.loc[subj, obj] = np.nan

    print("Exporting to files...")

    # save matrix to file as a pandas dataframe
    matrix.to_csv("../bank_files/cn_full_matrix.csv")

    # save keys to numpy files
    np.save("../bank_files/cn_full_subjects.npy", np.array(subject_list))
    np.save("../bank_files/cn_full_objects.npy", np.array(object_list))
    np.save("../bank_files/cn_full_verbs.npy", np.array(list(verb_index.keys())))

In [9]:
if MATRIX:
    # view dataframe (import)
    matrix = pd.read_csv("../bank_files/cn_full_matrix.csv", index_col=0)
    matrix.head()

In [10]:
if MATRIX:
    verb_list = np.load("../bank_files/cn_full_verbs.npy", allow_pickle=True)
    print(matrix.shape)
    print(len(verb_list))

### Clean up the NP files (whitelist)

In [14]:
if CLEAN_NP:
    from spellchecker import SpellChecker

    print("Loading badwords.txt...")

    with open("../bank_files/badwords.txt", "r", encoding="utf-8") as f:
        bad_words = set([l.strip() for l in f.readlines()])


    # clean up the SUBJECTS.npy file to only include words that are in the matrix
    print("Loading npy files...")
    subjects = np.load("../bank_files/SUBJECTS.npy", allow_pickle=True)
    objects = np.load("../bank_files/OBJECTS.npy", allow_pickle=True)
    verbs = np.load("../bank_files/VERBS.npy", allow_pickle=True)

    # remove misspelled entries, NSFW, and un-PC bad words
    print("Cleaning up npy files...")
    spell = SpellChecker()
    cleaned_subjects = []
    cleaned_objects = []
    cleaned_verbs = []
    for word in tqdm(subjects):
        if word not in bad_words and spell.correction(word) == word:
            cleaned_subjects.append(word)
    for word in tqdm(objects):
        if word not in bad_words and spell.correction(word) == word:
            cleaned_objects.append(word)
    for word in tqdm(verbs):
        if word not in bad_words and spell.correction(word) == word:
            cleaned_verbs.append(word)

    print(f"Original SUBJECTS: {len(subjects)}, Cleaned: {len(cleaned_subjects)}")
    print(f"Original OBJECTS: {len(objects)}, Cleaned: {len(cleaned_objects)}")
    print(f"Original VERBS: {len(verbs)}, Cleaned: {len(cleaned_verbs)}")

    np.save("../bank_files/SUBJECTS_clean.npy", np.array(cleaned_subjects))
    np.save("../bank_files/OBJECTS_clean.npy", np.array(cleaned_objects))
    np.save("../bank_files/VERBS_clean.npy", np.array(cleaned_verbs))

Loading badwords.txt...
Loading npy files...
Cleaning up npy files...


100%|██████████| 1506/1506 [00:14<00:00, 103.94it/s]

Original SUBJECTS: 6018, Cleaned: 5344
Original OBJECTS: 4849, Cleaned: 3191
Original VERBS: 1506, Cleaned: 1378


In [15]:
# export to text file for funsies and extra cleaning
if CLEAN_NP:
    with open("../bank_files/SUBJECTS_txt.txt", "w+", encoding="utf-8") as f:
        f.write("\n".join(cleaned_subjects))
    with open("../bank_files/OBJECTS_txt.txt", "w+", encoding="utf-8") as f:
        f.write("\n".join(cleaned_objects))
    with open("../bank_files/VERBS_txt.txt", "w+", encoding="utf-8") as f:
        f.write("\n".join(cleaned_verbs))